In [3]:
pip install kafka-python yfinance


  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ --------------------- 0.8/1.7 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 4.7 MB/s eta 0:00:00
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\Antara\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [1]:
import json
import time
from datetime import datetime, timezone
import yfinance as yf
from kafka import KafkaProducer

In [11]:
BOOTSTRAP_SERVERS = ["localhost:9092"]
TOPIC = "stock_prices_v2"
TICKERS = ["AAPL", "MSFT", "GOOGL"]
POLL_INTERVAL_SECONDS = 5

In [12]:
def build_producer () -> KafkaProducer:
    return KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )

In [13]:
def fetch_price(ticker: str) -> dict | None:
    try:
        info = yf.Ticker(ticker).fast_info
        return {
            "ticker": ticker,
            "price": float(info["lastPrice"]),
            "fetched_at": datetime.now(timezone.utc).isoformat(),
        }
    except Exception as e:
        print(f"[producer] could not fetch {ticker}: {e}")
        return None

In [14]:
def main():
    producer = build_producer()
    print(f"[producer] sending to topic '{TOPIC}' every {POLL_INTERVAL_SECONDS}s. Ctrl+C to stop.")
    try:
        while True:
            for ticker in TICKERS:
                record = fetch_price(ticker)
                if record:
                    producer.send(TOPIC, key=ticker.encode("utf-8"), value=record)
                    print(f"[producer] sent {record}")
            producer.flush()
            time.sleep(POLL_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        print("\n[producer] stopped.")
    finally:
        producer.close()

In [16]:
if __name__ == "__main__":
    main()

C:\Users\Antara\AppData\Local\Temp\ipykernel_45156\1468923172.py:2: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = build_producer()


[producer] sending to topic 'stock_prices_v2' every 5s. Ctrl+C to stop.
[producer] sent {'ticker': 'AAPL', 'price': 312.7799987792969, 'fetched_at': '2026-07-08T16:50:41.410326+00:00'}
[producer] sent {'ticker': 'MSFT', 'price': 382.80999755859375, 'fetched_at': '2026-07-08T16:50:41.995184+00:00'}
[producer] sent {'ticker': 'GOOGL', 'price': 361.0799865722656, 'fetched_at': '2026-07-08T16:50:42.315443+00:00'}
[producer] sent {'ticker': 'AAPL', 'price': 312.7799987792969, 'fetched_at': '2026-07-08T16:50:47.597067+00:00'}
[producer] sent {'ticker': 'MSFT', 'price': 382.80999755859375, 'fetched_at': '2026-07-08T16:50:47.778829+00:00'}
[producer] sent {'ticker': 'GOOGL', 'price': 361.0799865722656, 'fetched_at': '2026-07-08T16:50:48.012880+00:00'}
[producer] sent {'ticker': 'AAPL', 'price': 312.93011474609375, 'fetched_at': '2026-07-08T16:50:53.472811+00:00'}
[producer] sent {'ticker': 'MSFT', 'price': 382.62200927734375, 'fetched_at': '2026-07-08T16:50:53.803720+00:00'}
[producer] sent {'